# Pretraining Data Pipeline

তিনটি demo, সবগুলোই খাঁটি Python/স্ট্যান্ডার্ড-লাইব্রেরি (এই পাঠটি document-স্তরের অ্যালগরিদম নিয়ে, নিউরাল নেট নয়, তাই এখানে PyTorch নেই):

  1. একটি heuristic quality filter (খুব ছোট / অতিরিক্ত symbol-বহুল / অতিরিক্ত পুনরাবৃত্তিমূলক) যা "web document"-এর একটি toy pool-এ প্রয়োগ করা হয়েছে।
  2. Hashing দিয়ে exact deduplication — দ্রুত, নির্ভুল, কিন্তু byte-অভিন্ন ছাড়া আর কিছুই দেখে না।
  3. স্ক্র্যাচ থেকে লেখা একটি MinHash near-duplicate সনাক্তকারী — ছোট signature থেকে Jaccard similarity আনুমানিক করে এবং near-duplicate document-গুলোকে (মাত্র কয়েকটি শব্দে ভিন্ন) সঠিকভাবে চিহ্নিত করে, যেগুলো exact-hash dedup সম্পূর্ণ মিস করে।

**চালানোর নিয়ম:** কোষগুলো উপরে থেকে নিচে (Run All) চালান। প্রতিটি অংশের demo তার নিজের কোষেই চলে, আর শেষ কোষের `main()` পুরো রানটি একসাথে আরেকবার চালায়।

In [ ]:
import hashlib
import random
from collections import Counter

random.seed(0)

## ডকুমেন্ট pool (setup data)

HTML extraction-এ বেঁচে যাওয়া "document"-এর প্রতিনিধি — অর্থাৎ কাঁচা HTML নয়, ইতিমধ্যেই সাধারণ text। এর মধ্যে রয়েছে: সাধারণ document, exact duplicate, NEAR-duplicate (কয়েকটি শব্দ বদলানো), এবং স্পষ্টতই নিম্ন-মানের document (খুব ছোট, spammy/symbol-বহুল, পুনরাবৃত্তিমূলক)।

In [ ]:
# ---------------------------------------------------------------------------
# Toy document pool। HTML extraction-এ বেঁচে যাওয়া "document"-এর প্রতিনিধি --
# অর্থাৎ কাঁচা HTML নয়, ইতিমধ্যেই সাধারণ text। রয়েছে: সাধারণ document,
# exact duplicate, NEAR-duplicate (কয়েকটি শব্দ বদলানো), এবং স্পষ্টতই
# নিম্ন-মানের document (খুব ছোট, spammy/symbol-বহুল, পুনরাবৃত্তিমূলক)।
# ---------------------------------------------------------------------------

DOCS = {
    "wiki_forest_1": (
        "The boreal forest, also known as taiga, is the world's largest land "
        "biome. It is characterized by coniferous trees such as pine, spruce, "
        "and fir, and experiences long, cold winters with short, mild summers. "
        "Many species of birds migrate to the taiga only during the summer "
        "breeding season."
    ),
    # উপরের ডকুমেন্টের exact duplicate -- যেমন, একটি syndicated mirror copy।
    "mirror_forest_1": (
        "The boreal forest, also known as taiga, is the world's largest land "
        "biome. It is characterized by coniferous trees such as pine, spruce, "
        "and fir, and experiences long, cold winters with short, mild summers. "
        "Many species of birds migrate to the taiga only during the summer "
        "breeding season."
    ),
    # NEAR-duplicate: একই document, কিন্তু কয়েকটি শব্দ বদলানো/ঢোকানো (যেন
    # দ্বিতীয় একটি সাইট হালকা এডিটিং করে পুনঃপ্রকাশ করেছে)। Exact hash dedup
    # এটা ধরবে না -- byte ভিন্ন -- কিন্তু MinHash ধরবে।
    "reprint_forest_1": (
        "The boreal forest, also called taiga, is the world's largest "
        "terrestrial biome. It is characterized by coniferous trees such as "
        "pine, spruce, and fir, and experiences long, harsh winters with "
        "short, cool summers. Many species of birds migrate to the taiga "
        "only during the brief summer breeding season."
    ),
    "wiki_ocean_1": (
        "The Pacific Ocean is the largest and deepest of Earth's five ocean "
        "basins. It extends from the Arctic Ocean in the north to the "
        "Southern Ocean in the south, and is bounded by Asia and Australia "
        "in the west and the Americas in the east."
    ),
    "blog_cooking_1": (
        "To make a simple tomato sauce, start by sauteing chopped garlic and "
        "onion in olive oil until fragrant. Add crushed tomatoes, a pinch of "
        "salt, and a few basil leaves, then let it simmer for twenty minutes "
        "until the sauce thickens and the flavors meld together."
    ),
    "news_article_1": (
        "City officials announced today that the downtown bridge renovation "
        "project will begin next month, with construction expected to take "
        "approximately eight months to complete. Traffic will be rerouted "
        "through the adjacent avenue during construction hours."
    ),
    # নিম্ন মানের: দরকারি training signal বহন করার পক্ষে অত্যন্ত ছোট।
    "spam_short_1": "Click here now!!!",
    # নিম্ন মানের: অত্যন্ত উচ্চ symbol-to-word অনুপাত -- সম্ভবত HTML extraction-এ
    # বেঁচে যাওয়া spam/markup অবশিষ্টাংশ।
    "spam_symbols_1": "!!! BUY NOW $$$ >>> #1 DEAL *** LIMITED TIME <<< 50% OFF !!! $$$ ###",
    # নিম্ন মানের: অত্যন্ত পুনরাবৃত্তিমূলক templated/boilerplate text।
    "spam_repeat_1": ("buy now buy now buy now " * 8).strip(),
    "academic_abstract_1": (
        "This paper presents a comprehensive empirical study of gradient "
        "descent optimization methods for large-scale neural network "
        "training, evaluating convergence behavior across a range of "
        "learning rate schedules and batch sizes."
    ),
}

## 1. Heuristic quality filter

প্রতিটি document-এ সস্তা, নিয়ম-ভিত্তিক চেক চালানো হয় — বাস্তব pipeline (CCNet, Gopher/MassiveText, RefinedWeb) কোনো দামি classifier জড়ানোর আগে এটিই করে। নিচের কোষে `quality_check` সংজ্ঞায়িত এবং toy document pool-এর উপর demo চালানো হয়েছে।

In [ ]:
# ---------------------------------------------------------------------------
# 1. Heuristic quality filter -- সস্তা, নিয়ম-ভিত্তিক মান-পরীক্ষা
# ---------------------------------------------------------------------------

def quality_check(text, min_words=8, max_symbol_ratio=0.28, max_repeat_frac=0.4):
    """(passed: bool, reason: str) রিটার্ন করে। বাস্তব pipeline (CCNet,
    Gopher/MassiveText, RefinedWeb) কোনো দামি classifier জড়ানোর আগে প্রতিটি
    document-এ যেসব সস্তা, নিয়ম-ভিত্তিক চেক চালায়, সেগুলোরই প্রতিফলন।"""
    words = text.split()
    num_words = len(words)

    if num_words < min_words:
        return False, f"too short ({num_words} words < {min_words})"

    # Symbol-to-word অনুপাত: alphanumeric বা whitespace নয় এমন character-এর
    # সংখ্যা, word-সংখ্যার সাপেক্ষে। Spammy/markup-অবশিষ্ট text-এ punctuation
    # ও symbol-ই প্রাধান্য পায়।
    num_symbols = sum(1 for ch in text if not ch.isalnum() and not ch.isspace())
    symbol_ratio = num_symbols / max(num_words, 1)
    if symbol_ratio > max_symbol_ratio:
        return False, f"symbol-to-word ratio too high ({symbol_ratio:.2f} > {max_symbol_ratio})"

    # Repetition: সবচেয়ে বেশি ব্যবহৃত একটি শব্দ মোট word-occurrence-এর কত
    # ভগ্নাংশ দখল করেছে। Templated/স্বয়ংক্রিয়-উৎপন্ন spam একই phrase বারবার
    # পুনরাবৃত্তি করে; স্বাভাবিক গদ্য তা করে না।
    counts = Counter(words)
    most_common_frac = counts.most_common(1)[0][1] / num_words
    if most_common_frac > max_repeat_frac:
        return False, f"too repetitive (top word = {most_common_frac:.0%} of all words)"

    return True, "passed"


def quality_filter_demo():
    print("=" * 70)
    print("1. HEURISTIC QUALITY FILTER")
    print("=" * 70)
    kept, rejected = [], []
    for name, text in DOCS.items():
        passed, reason = quality_check(text)
        (kept if passed else rejected).append(name)
        status = "KEEP  " if passed else "REJECT"
        print(f"  [{status}] {name:22s} {reason}")

    print(f"\n-> Kept {len(kept)}/{len(DOCS)} documents. Rejected: {rejected}")
    print("   Note the three 'spam_*' documents were all caught by simple,")
    print("   cheap rules -- no neural network needed for this pass. This is")
    print("   exactly why heuristic filtering runs FIRST, over the entire raw")
    print("   crawl, before any more expensive classifier-based filtering.")
    return kept


# এই অংশের demo: এখানেই চালানো হয়, ফলে notebook উপরে থেকে নিচে চালালেই কাজ করে।
kept = quality_filter_demo()

## 2. Exact deduplication via hashing

নরমালাইজড text-এর উপর SHA-1 hash চালিয়ে byte-অভিন্ন duplicate ধরা হয় — সস্তা ও সম্পূর্ণ নির্ভুল, কিন্তু near-duplicate অদৃশ্য থাকে। `exact_dedup_demo` অংশ 1-এ রাখা (kept) document-গুলোর উপর চলে।

In [ ]:
# ---------------------------------------------------------------------------
# 2. Exact deduplication via hashing -- hash দিয়ে হুবহু ডুপ্লিকেট বাদ
# ---------------------------------------------------------------------------

def normalize(text):
    """Lowercase + whitespace সংকোচন, যাতে তুচ্ছ formatting পার্থক্য
    (অতিরিক্ত space, capital letter) exact-hash ম্যাচিং নষ্ট না করে।"""
    return " ".join(text.lower().split())


def exact_dedup(docs):
    """নরমালাইজড text-এর উপর SHA-1 ব্যবহার করে (unique_docs, duplicate_pairs)
    রিটার্ন করে।"""
    seen_hashes = {}
    unique_docs = {}
    duplicate_pairs = []
    for name, text in docs.items():
        h = hashlib.sha1(normalize(text).encode("utf-8")).hexdigest()
        if h in seen_hashes:
            duplicate_pairs.append((seen_hashes[h], name))
        else:
            seen_hashes[h] = name
            unique_docs[name] = text
    return unique_docs, duplicate_pairs


def exact_dedup_demo(kept_docs):
    print("\n" + "=" * 70)
    print("2. EXACT DEDUPLICATION (SHA-1 HASHING)")
    print("=" * 70)
    docs = {name: DOCS[name] for name in kept_docs}
    unique_docs, dup_pairs = exact_dedup(docs)

    print(f"Documents going in:  {len(docs)}")
    print(f"Exact duplicates found: {dup_pairs if dup_pairs else 'none'}")
    print(f"Documents remaining after exact dedup: {len(unique_docs)}")
    print("\n-> 'mirror_forest_1' is a byte-identical copy of 'wiki_forest_1' and")
    print("   was correctly caught. But 'reprint_forest_1' -- the SAME article")
    print("   with a handful of words changed -- has a different hash entirely,")
    print("   so exact dedup lets it straight through. That's exactly the gap")
    print("   near-duplicate detection (part 3) exists to close.")
    return unique_docs


# এই অংশের demo: অংশ 1-এর kept ফলাফল ব্যবহার করে।
unique_docs = exact_dedup_demo(kept)

## 3. MinHash near-duplicate detection

Word n-gram shingle-এর MinHash signature থেকে Jaccard similarity আনুমানিক করা হয় — exact dedup যাকে মিস করে, এটা ঠিক তাকে ধরে: কয়েকটি শব্দ বদলানো হলেও `reprint_forest_1`-এর মতো প্রায়-ডুপ্লিকেট সঠিকভাবে চিহ্নিত হয়।

In [ ]:
# ---------------------------------------------------------------------------
# 3. MinHash near-duplicate detection -- প্রায়-ডুপ্লিকেট সনাক্তকরণ
# ---------------------------------------------------------------------------

def shingles(text, n=3):
    """Word n-gram shingle -- MinHash-এর তুলনার আদর্শ একক। Character shingle-ও
    চলে; word shingle পরীক্ষা করে পড়তে সহজ।"""
    words = normalize(text).split()
    if len(words) < n:
        return {" ".join(words)}
    return {" ".join(words[i:i + n]) for i in range(len(words) - n + 1)}


def true_jaccard(set_a, set_b):
    return len(set_a & set_b) / len(set_a | set_b)


def minhash_signature(shingle_set, num_hashes, salts):
    """একটি MinHash signature = num_hashes সংখ্যক স্বাধীন 'shingle set-এর উপর
    ন্যূনতম hash মান' গণনা। প্রতিটি salt-কে আলাদাভাবে MD5-এ মিশিয়ে একটি স্বাধীন
    এলোমেলো hash function অনুকরণ করা হয় -- হাতে num_hashes সংখ্যক আলাদা hash
    algorithm না থাকলে এটিই প্রচলিত সস্তা কৌশল।"""
    signature = []
    for salt in salts:
        min_val = min(
            int(hashlib.md5(f"{salt}:{shingle}".encode("utf-8")).hexdigest(), 16)
            for shingle in shingle_set
        )
        signature.append(min_val)
    return signature


def estimated_jaccard_from_signatures(sig_a, sig_b):
    """ম্যাচিং signature অবস্থানের ভগ্নাংশ = প্রকৃত Jaccard similarity-র unbiased
    estimator (এটিই README-এর মূল MinHash ধর্ম:
    P(minhash_h(A) == minhash_h(B)) == Jaccard(A, B) একটি hash-এর জন্য)।"""
    matches = sum(1 for a, b in zip(sig_a, sig_b) if a == b)
    return matches / len(sig_a)


def minhash_dedup_demo(unique_docs):
    print("\n" + "=" * 70)
    print("3. NEAR-DUPLICATE DETECTION VIA MINHASH")
    print("=" * 70)

    num_hashes = 128
    salts = list(range(num_hashes))

    doc_shingles = {name: shingles(text) for name, text in unique_docs.items()}
    doc_signatures = {
        name: minhash_signature(s, num_hashes, salts) for name, s in doc_shingles.items()
    }

    print(f"Computing {num_hashes}-element MinHash signatures over 3-word shingles "
          f"for {len(unique_docs)} documents (post exact-dedup).\n")

    names = list(unique_docs.keys())
    threshold = 0.5   # estimated Jaccard এই সীমা ছাড়ালে near-duplicate হিসেবে চিহ্নিত হবে
    print(f"{'doc A':22s}{'doc B':22s}{'true Jaccard':>14}{'MinHash est.':>14}{'flagged?':>10}")
    flagged_pairs = []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = names[i], names[j]
            tj = true_jaccard(doc_shingles[a], doc_shingles[b])
            ej = estimated_jaccard_from_signatures(doc_signatures[a], doc_signatures[b])
            is_flagged = ej > threshold
            if is_flagged:
                flagged_pairs.append((a, b))
            # শুধুমাত্র অর্থবহ সাদৃশ্যযুক্ত pair-গুলো ছাপা হয়, যাতে টেবিলটি
            # পড়তে সুবিধা হয় -- অধিকাংশ cross-topic pair-এ দুটোই প্রায় 0।
            if tj > 0.05 or ej > 0.05:
                print(f"{a:22s}{b:22s}{tj:>14.3f}{ej:>14.3f}{'YES' if is_flagged else 'no':>10}")

    print(f"\nFlagged as near-duplicates (estimated Jaccard > {threshold}): {flagged_pairs}")
    print("\n-> 'wiki_forest_1' and 'reprint_forest_1' share no identical bytes and")
    print("   were completely invisible to exact-hash dedup in part 2, yet MinHash's")
    print("   signature-based ESTIMATE of their Jaccard similarity lands close to the")
    print("   true value computed directly from the shingle sets, and correctly")
    print("   clears the near-duplicate threshold. This is exactly how billion-document")
    print("   pipelines catch republished/lightly-edited content: comparing a handful")
    print("   of small integers per document instead of the full text of every pair.")


def main():
    kept = quality_filter_demo()
    unique_docs = exact_dedup_demo(kept)
    minhash_dedup_demo(unique_docs)


# এই অংশের demo (part 2-এর ফলাফলের উপর ভিত্তি করে)।
minhash_dedup_demo(unique_docs)

In [ ]:
main()